In [1]:
# CELL 1: Installation
!pip install -q datasets evaluate rouge_score nltk sacrebleu
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

print("Installation complete!")

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 9.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 122.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# CELL 2: Import Libraries
import torch
import pandas as pd
import numpy as np
import json
from datetime import datetime
from typing import Dict, List
from abc import ABC, abstractmethod

from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments, TrainerCallback
import evaluate

print("Libraries imported!")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Libraries imported!
CUDA available: True
GPU: Tesla T4


In [3]:
# CELL 3: HuggingFace Login
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)
    print("Logged into HuggingFace")
except Exception as e:
    print(f"HF_TOKEN not found: {e}")
    login()

HF_TOKEN not found: Secret HF_TOKEN does not exist.


In [4]:
# CELL 4: Metrics Callback
class MetricsCallback(TrainerCallback):
    """Track training metrics"""

    def __init__(self):
        self.train_losses = []
        self.eval_losses = []
        self.learning_rates = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            if 'loss' in logs:
                self.train_losses.append({
                    'step': state.global_step,
                    'loss': logs['loss']
                })
            if 'eval_loss' in logs:
                self.eval_losses.append({
                    'step': state.global_step,
                    'loss': logs['eval_loss']
                })
            if 'learning_rate' in logs:
                self.learning_rates.append({
                    'step': state.global_step,
                    'lr': logs['learning_rate']
                })

In [5]:
# CELL 5: Dataset Processor
class DatasetProcessor:
    """Bengali Empathetic Conversations Preprocessor"""

    def __init__(self, dataset_path: str, tokenizer):
        self.dataset_path = dataset_path
        self.tokenizer = tokenizer
        self.stats = {}

    def load_and_preprocess(self):
        """Load CSV and create formatted text strings"""
        print(f"\n{'='*60}")
        print("📂 Loading Dataset")
        print(f"{'='*60}\n")

        df = pd.read_csv(self.dataset_path)

        print(f"Dataset columns: {df.columns.tolist()}")
        print(f"Dataset shape: {df.shape}")
        print(f"First row sample:\n{df.head(1)}\n")

        # Map columns (case-insensitive)
        col_map = {col.lower().strip(): col for col in df.columns}

        # Find correct column names
        topic_col = None
        title_col = None
        question_col = None
        answer_col = None

        for key, col in col_map.items():
            if 'topic' in key:
                topic_col = col
            elif 'title' in key:
                title_col = col
            elif 'question' in key and 'title' not in key:
                question_col = col
            elif 'answer' in key:
                answer_col = col

        print(f"Mapped columns:")
        print(f"  Topic: {topic_col}")
        print(f"  Title: {title_col}")
        print(f"  Question: {question_col}")
        print(f"  Answer: {answer_col}\n")

        if not question_col or not answer_col:
            raise ValueError(f"Missing required columns! Found: {df.columns.tolist()}")

        formatted_data = []
        skipped = 0
        lengths = []

        for idx, row in df.iterrows():
            # Extract and clean data
            topic = str(row[topic_col]).strip() if topic_col and pd.notna(row[topic_col]) else ""
            title = str(row[title_col]).strip() if title_col and pd.notna(row[title_col]) else ""
            question = str(row[question_col]).strip() if pd.notna(row[question_col]) else ""
            answer = str(row[answer_col]).strip() if pd.notna(row[answer_col]) else ""

            # Skip invalid entries
            if not question or not answer or len(answer) < 10:
                skipped += 1
                continue

            # Build user prompt with context
            context_parts = []
            if topic and topic != 'nan':
                context_parts.append(f"Topic: {topic}")
            if title and title != 'nan':
                context_parts.append(f"Title: {title}")
            context_parts.append(f"Question: {question}")

            user_content = "\n".join(context_parts)

            # Apply chat template to create formatted text
            try:
                formatted_text = self.tokenizer.apply_chat_template(
                    [
                        {"role": "user", "content": user_content},
                        {"role": "assistant", "content": answer}
                    ],
                    tokenize=False,
                    add_generation_prompt=False
                )

                formatted_data.append({
                    "text": formatted_text,
                    "user_content": user_content,
                    "answer": answer
                })

                lengths.append(len(formatted_text))

            except Exception as e:
                print(f"Error formatting row {idx}: {e}")
                skipped += 1
                continue

        print(f"Formatted: {len(formatted_data)} examples")
        print(f"Skipped: {skipped} examples")
        print(f"Avg text length: {np.mean(lengths):.0f} chars")
        print(f"Max text length: {max(lengths):.0f} chars\n")

        # Create dataset and splits
        dataset = Dataset.from_list(formatted_data)
        subset_60 = dataset.train_test_split(test_size=0.6, seed=42)['train']
        train_test = subset_60.train_test_split(test_size=0.1, seed=42)
        val_test = train_test['test'].train_test_split(test_size=0.5, seed=42)

        self.stats = {
            'total': len(formatted_data),
            'used': len(subset_60),
            'train': len(train_test['train']),
            'val': len(val_test['train']),
            'test': len(val_test['test']),
            'avg_length': np.mean(lengths),
            'max_length': max(lengths)
        }

        return train_test['train'], val_test['train'], val_test['test']

In [6]:
# CELL 6: Strategy Pattern
class FineTuningStrategy(ABC):
    @abstractmethod
    def configure_model(self, model):
        pass

    @abstractmethod
    def get_config_info(self) -> Dict:
        pass

class UnslothLoRAStrategy(FineTuningStrategy):
    """Unsloth-optimized LoRA"""

    def __init__(self, r=8, lora_alpha=8, lora_dropout=0.05):
        self.r = r
        self.lora_alpha = lora_alpha
        self.lora_dropout = lora_dropout

    def configure_model(self, model):
        model = FastLanguageModel.get_peft_model(
            model,
            r=self.r,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
            lora_alpha=self.lora_alpha,
            lora_dropout=self.lora_dropout,
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=42,
            use_rslora=False,
        )
        return model

    def get_config_info(self) -> Dict:
        return {
            "strategy": "Unsloth-LoRA",
            "r": self.r,
            "lora_alpha": self.lora_alpha,
            "lora_dropout": self.lora_dropout
        }

class StandardLoRAStrategy(FineTuningStrategy):
    """Standard LoRA"""

    def __init__(self, r=8, lora_alpha=8, lora_dropout=0.1):
        self.r = r
        self.lora_alpha = lora_alpha
        self.lora_dropout = lora_dropout

    def configure_model(self, model):
        from peft import LoraConfig, get_peft_model

        config = LoraConfig(
            r=self.r,
            lora_alpha=self.lora_alpha,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
            lora_dropout=self.lora_dropout,
            bias="none",
            task_type="CAUSAL_LM"
        )
        return get_peft_model(model, config)

    def get_config_info(self) -> Dict:
        return {
            "strategy": "Standard-LoRA",
            "r": self.r,
            "lora_alpha": self.lora_alpha,
            "lora_dropout": self.lora_dropout
        }

In [7]:
# CELL 7: Main Fine-Tuner
class LLAMAFineTuner:
    """Production fine-tuning pipeline"""

    def __init__(self, model, tokenizer, strategy: FineTuningStrategy):
        self.base_model = model
        self.tokenizer = tokenizer
        self.strategy = strategy
        self.model = None
        self.trainer = None
        self.metrics_callback = MetricsCallback()
        self.experiment_id = datetime.now().strftime("%Y%m%d_%H%M%S")

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.logs = {
            "experiment_id": self.experiment_id,
            "model_name": "Meta-Llama-3.1-8B-Instruct",
            "lora_config": strategy.get_config_info(),
            "train_losses": [],
            "eval_losses": [],
            "metrics": {},
            "timestamp": datetime.now().isoformat()
        }

    def prepare_model(self):
        """Apply LoRA configuration"""
        print(f"\n{'='*60}")
        print(f"🔧 Applying {self.strategy.get_config_info()['strategy']}")
        print(f"{'='*60}\n")

        self.model = self.strategy.configure_model(self.base_model)

        total = sum(p.numel() for p in self.model.parameters())
        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)

        print(f"Parameters:")
        print(f"  Total: {total:,}")
        print(f"  Trainable: {trainable:,}")
        print(f"  Trainable %: {100 * trainable / total:.2f}%\n")

        self.logs['total_params'] = total
        self.logs['trainable_params'] = trainable

        return self.model

    def setup_trainer(self, train_dataset, val_dataset,
                     output_dir="./results",
                     num_train_epochs=1,
                     per_device_batch_size=1,
                     gradient_accumulation_steps=4,
                     learning_rate=2e-4,
                     max_seq_length=1024,
                     warmup_ratio=0.1,
                     eval_steps=50):
        """Setup SFTTrainer"""

        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_train_epochs,
            per_device_train_batch_size=per_device_batch_size,
            per_device_eval_batch_size=per_device_batch_size,
            gradient_accumulation_steps=gradient_accumulation_steps,
            warmup_ratio=warmup_ratio,
            learning_rate=learning_rate,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            logging_steps=20,
            eval_strategy="steps",
            eval_steps=eval_steps,
            save_strategy="steps",
            save_steps=eval_steps,
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            seed=42,
            report_to="none",
            gradient_checkpointing=True,
            max_grad_norm=0.3,
            group_by_length=True,
            dataloader_num_workers=2,
        )

        self.trainer = SFTTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            max_seq_length=max_seq_length,
            dataset_text_field="text",
            packing=True,
            args=training_args,
            callbacks=[self.metrics_callback],
        )

        print(f"Trainer configured:")
        print(f"  Epochs: {num_train_epochs}")
        print(f"  Batch size: {per_device_batch_size}")
        print(f"  Gradient accumulation: {gradient_accumulation_steps}")
        print(f"  Effective batch: {per_device_batch_size * gradient_accumulation_steps}")
        print(f"  Learning rate: {learning_rate}")
        print(f"  Max sequence length: {max_seq_length}\n")

        return self.trainer

    def train(self):
        """Execute training"""
        print(f"\n{'='*60}")
        print("Training Started")
        print(f"{'='*60}\n")

        train_result = self.trainer.train()

        self.logs["train_losses"] = self.metrics_callback.train_losses
        self.logs["eval_losses"] = self.metrics_callback.eval_losses
        self.logs["final_train_loss"] = train_result.training_loss if hasattr(train_result, 'training_loss') else None
        self.logs["training_time"] = train_result.metrics.get("train_runtime", 0)

        print(f"\nTraining Complete!")
        if hasattr(train_result, 'training_loss'):
            print(f"  Final Training Loss: {train_result.training_loss:.4f}")
        print(f"  Training Time: {train_result.metrics.get('train_runtime', 0):.2f}s\n")

        return train_result

    def evaluate(self, eval_dataset=None):
        """Evaluate model"""
        print(f"\n{'='*60}")
        print("Evaluation")
        print(f"{'='*60}\n")

        eval_result = self.trainer.evaluate(eval_dataset)

        self.logs["final_eval_loss"] = eval_result.get("eval_loss", None)
        self.logs["metrics"].update(eval_result)

        print(f"Evaluation Loss: {eval_result.get('eval_loss', 'N/A'):.4f}\n")

        return eval_result

    def save_model(self, output_dir="./fine_tuned_model"):
        """Save model"""
        print(f"\nSaving to {output_dir}...")

        self.model.save_pretrained(output_dir)
        self.tokenizer.save_pretrained(output_dir)

        with open(f"{output_dir}/training_logs.json", "w", encoding='utf-8') as f:
            json.dump(self.logs, f, indent=2, ensure_ascii=False)

        print(f"Saved!\n")

    def generate_response(self, prompt: str, max_new_tokens=256, temperature=0.7):
        """Generate response"""
        FastLanguageModel.for_inference(self.model)

        messages = [{"role": "user", "content": prompt}]
        inputs = self.tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to("cuda")

        outputs = self.model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
        )

        full_response = self.tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]

        # Extract assistant response
        assistant_marker = "<|start_header_id|>assistant<|end_header_id|>"
        if assistant_marker in full_response:
            response = full_response.split(assistant_marker)[-1].split("<|eot_id|>")[0].strip()
        else:
            response = self.tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

        return response

In [8]:
# CELL 8: Evaluator
class Evaluator:
    """Compute metrics"""

    def __init__(self):
        self.rouge = evaluate.load('rouge')
        self.bleu = evaluate.load('sacrebleu')

    def compute_metrics(self, predictions: List[str], references: List[str]) -> Dict:
        """Calculate ROUGE and BLEU"""
        rouge_scores = self.rouge.compute(
            predictions=predictions,
            references=references,
            use_stemmer=True
        )

        bleu_scores = self.bleu.compute(
            predictions=predictions,
            references=[[ref] for ref in references]
        )

        return {
            'rouge1': rouge_scores['rouge1'],
            'rouge2': rouge_scores['rouge2'],
            'rougeL': rouge_scores['rougeL'],
            'bleu': bleu_scores['score'] / 100.0,
        }

    def compute_perplexity(self, model, tokenizer, dataset, max_samples=100):
        """Calculate perplexity"""
        model.eval()
        total_loss = 0
        count = 0

        print(f"Computing perplexity on {min(max_samples, len(dataset))} samples...")

        with torch.no_grad():
            for i, example in enumerate(dataset):
                if i >= max_samples:
                    break

                text = example.get('text', '')
                if not text:
                    continue

                inputs = tokenizer(
                    text,
                    return_tensors='pt',
                    truncation=True,
                    max_length=512
                ).to(model.device)

                try:
                    outputs = model(**inputs, labels=inputs['input_ids'])
                    total_loss += outputs.loss.item()
                    count += 1
                except Exception as e:
                    continue

        if count == 0:
            return float('inf')

        avg_loss = total_loss / count
        perplexity = np.exp(avg_loss)

        print(f"Perplexity: {perplexity:.4f}\n")

        return perplexity

In [9]:
# CELL 9: Load Model
print(f"\n{'='*60}")
print("Loading LLaMA 3.1-8B-Instruct")
print(f"{'='*60}\n")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

print("Model loaded!")


Loading LLaMA 3.1-8B-Instruct

==((====))==  Unsloth 2026.1.1: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Model loaded!


In [12]:
# CELL 10: Load Dataset
dataset_path = "/content/BengaliEmpatheticConversationsCorpus .csv"

processor = DatasetProcessor(dataset_path, tokenizer)
train_ds, val_ds, test_ds = processor.load_and_preprocess()

print(f"\nDataset Statistics:")
print(f"  Train: {len(train_ds)}")
print(f"  Val: {len(val_ds)}")
print(f"  Test: {len(test_ds)}")
print(f"  Total: {processor.stats['total']}\n")


📂 Loading Dataset

Dataset columns: ['Topics', 'Question-Title', 'Questions', 'Answers']
Dataset shape: (38233, 4)
First row sample:
               Topics                      Question-Title  \
0  পারিবারিক দ্বন্দ্ব  মা ও স্ত্রীর মধ্যে মতানৈক্য বৃদ্ধি   

                                           Questions  \
0   আমার স্ত্রী এবং মায়ের মধ্যে টানটান মতবিরোধ চ...   

                                             Answers  
0   আপনি যা বর্ণনা করছেন তাকে মনোবিজ্ঞানীরা "ত্রি...  

Mapped columns:
  Topic: Topics
  Title: Question-Title
  Question: Questions
  Answer: Answers

Formatted: 37555 examples
Skipped: 678 examples
Avg text length: 474 chars
Max text length: 5985 chars


Dataset Statistics:
  Train: 13519
  Val: 751
  Test: 752
  Total: 37555



In [13]:
# CELL 11: Configure Strategy
strategy = UnslothLoRAStrategy(r=16, lora_alpha=16, lora_dropout=0.05)

In [14]:
# CELL 12: Initialize Fine-Tuner
finetuner = LLAMAFineTuner(model, tokenizer, strategy)
finetuner.prepare_model()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.



🔧 Applying Unsloth-LoRA



Unsloth 2026.1.1 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Parameters:
  Total: 4,582,543,360
  Trainable: 41,943,040
  Trainable %: 0.92%



PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
       

In [15]:
# CELL 13: Setup Trainer
finetuner.setup_trainer(
    train_dataset=train_ds,
    val_dataset=val_ds,
    output_dir="./llama_bengali_checkpoints",
    num_train_epochs=0.5,
    per_device_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    max_seq_length=1024,
    warmup_ratio=0.03,
    eval_steps=200
)

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/13519 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/751 [00:00<?, ? examples/s]

Trainer configured:
  Epochs: 0.5
  Batch size: 4
  Gradient accumulation: 2
  Effective batch: 8
  Learning rate: 0.0002
  Max sequence length: 1024



In [16]:
# CELL 14: Train
train_results = finetuner.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.



Training Started



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 13,519 | Num Epochs = 1 | Total steps = 845
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
200,0.531300,0.593748
400,0.519400,0.562836
600,0.480100,0.535315
800,0.481100,0.526285


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



Training Complete!
  Final Training Loss: 0.5615
  Training Time: 7809.95s



In [17]:
# CELL 15: Evaluate
eval_results = finetuner.evaluate()


Evaluation



Evaluation Loss: 0.5257



In [18]:
# CELL 16: Test Generation
print(f"\n{'='*60}")
print("Sample Generations")
print(f"{'='*60}\n")

test_prompts = [
    "Topic: পারিবারিক দ্বন্দ্ব\nTitle: মা ও স্ত্রীর মধ্যে মতানৈক্য\nQuestion: আমার স্ত্রী এবং মায়ের মধ্যে টানটান মতবিরোধ চলছে। আমি কীভাবে এই পরিস্থিতি সামলাব?",
    "Topic: মানসিক স্বাস্থ্য\nQuestion: আমি সবসময় দুশ্চিন্তায় থাকি। এটা কি স্বাভাবিক?",
    "Topic: ক্যারিয়ার\nQuestion: চাকরি ছেড়ে নিজের ব্যবসা শুরু করা কি ঠিক হবে?"
]

for i, prompt in enumerate(test_prompts, 1):
    print(f"Example {i}:")
    print(f"Prompt: {prompt[:80]}...")
    response = finetuner.generate_response(prompt, max_new_tokens=200)
    print(f"Response: {response}\n")
    print("-" * 60 + "\n")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Sample Generations

Example 1:
Prompt: Topic: পারিবারিক দ্বন্দ্ব
Title: মা ও স্ত্রীর মধ্যে মতানৈক্য
Question: আমার স্ত্...
Response: আমি মনে করি আপনার স্ত্রী এবং মায়ের মধ্যে একটি মতবিরোধ সম্পর্কে একটি আলোচনা করা উচিত। আমি মনে করি আপনার স্ত্রী এবং মায়ের মধ্যে একটি পার্থক্য হতে পারে যা আপনাকে একটি

------------------------------------------------------------

Example 2:
Prompt: Topic: মানসিক স্বাস্থ্য
Question: আমি সবসময় দুশ্চিন্তায় থাকি। এটা কি স্বাভাবিক...
Response: কিছু লোকের মানসিক স্বাস্থ্য স্বাভাবিক থাকে, কিন্তু যদি এটি তাদের জীবনের সাধারণ স্বাভাবিক থাকে না তবে আপনি একটি সমস্যার মুখোমুখি হতে পারেন। আপনি সাধারণত দুশ্চিন্তায় থাকত

------------------------------------------------------------

Example 3:
Prompt: Topic: ক্যারিয়ার
Question: চাকরি ছেড়ে নিজের ব্যবসা শুরু করা কি ঠিক হবে?...
Response: আমি মনে করি এটা ঠিক হবে, কিন্তু আপনার সম্পত্তি সংরক্ষণ করতে হবে।

------------------------------------------------------------



In [ ]:
# CELL 17: Comprehensive Evaluation
print(f"\n{'='*60}")
print("Comprehensive Evaluation")
print(f"{'='*60}\n")

evaluator = Evaluator()

predictions = []
references = []
num_samples = min(50, len(test_ds))

print(f"Generating {num_samples} predictions...")
for i, example in enumerate(test_ds.select(range(num_samples))):
    if i % 10 == 0:
        print(f"  {i}/{num_samples}...")

    user_content = example.get('user_content', '')
    reference = example.get('answer', '')

    if not user_content or not reference:
        continue

    try:
        pred = finetuner.generate_response(user_content, max_new_tokens=150)
        predictions.append(pred)
        references.append(reference)
    except Exception as e:
        continue

print(f"\nGenerated {len(predictions)} predictions\n")

if len(predictions) > 0:
    metrics = evaluator.compute_metrics(predictions, references)
    perplexity = evaluator.compute_perplexity(finetuner.model, tokenizer, test_ds, max_samples=100)

    print(f"{'='*60}")
    print("Final Metrics")
    print(f"{'='*60}")
    print(f"ROUGE-1:    {metrics['rouge1']:.4f}")
    print(f"ROUGE-2:    {metrics['rouge2']:.4f}")
    print(f"ROUGE-L:    {metrics['rougeL']:.4f}")
    print(f"BLEU:       {metrics['bleu']:.4f}")
    print(f"Perplexity: {perplexity:.4f}")
    print(f"{'='*60}\n")

    finetuner.logs['test_metrics'] = metrics
    finetuner.logs['test_perplexity'] = perplexity

In [20]:
# CELL 18: Save Everything
print(f"{'='*60}")
print("Saving Results")
print(f"{'='*60}\n")

finetuner.save_model("./llama_bengali_final")

# Save experiment results
experiment_results = {
    'experiment_id': finetuner.experiment_id,
    'model_name': 'Meta-Llama-3.1-8B-Instruct',
    'strategy': strategy.get_config_info()['strategy'],
    'lora_config': strategy.get_config_info(),
    'dataset_stats': processor.stats,
    'final_train_loss': finetuner.logs.get('final_train_loss'),
    'final_eval_loss': finetuner.logs.get('final_eval_loss'),
    'rouge1': metrics.get('rouge1') if 'metrics' in locals() else None,
    'rouge2': metrics.get('rouge2') if 'metrics' in locals() else None,
    'rougeL': metrics.get('rougeL') if 'metrics' in locals() else None,
    'bleu': metrics.get('bleu') if 'metrics' in locals() else None,
    'perplexity': perplexity if 'perplexity' in locals() else None,
    'total_params': finetuner.logs.get('total_params'),
    'trainable_params': finetuner.logs.get('trainable_params'),
    'timestamp': datetime.now().isoformat()
}

# LLAMAExperiments table
experiments_df = pd.DataFrame([experiment_results])
experiments_df.to_csv('llama_experiments.csv', index=False)

# GeneratedResponses table
if len(predictions) > 0:
    responses_data = []
    for i in range(min(20, len(predictions))):
        responses_data.append({
            'id': i + 1,
            'experiment_id': finetuner.experiment_id,
            'input_text': test_ds[i].get('user_content', '')[:200],
            'response_text': predictions[i],
            'reference': references[i],
            'timestamp': datetime.now().isoformat()
        })

    responses_df = pd.DataFrame(responses_data)
    responses_df.to_csv('generated_responses.csv', index=False, encoding='utf-8')

# Training history
with open('training_history.json', 'w', encoding='utf-8') as f:
    json.dump({
        'train_losses': finetuner.metrics_callback.train_losses,
        'eval_losses': finetuner.metrics_callback.eval_losses,
        'learning_rates': finetuner.metrics_callback.learning_rates
    }, f, indent=2)

print("All results saved!")
print("\nFiles:")
print("  - ./llama_bengali_final/ (model)")
print("  - ./llama_bengali_checkpoints/ (checkpoints)")
print("  - llama_experiments.csv")
print("  - generated_responses.csv")
print("  - training_history.json")

print("\n🎉 Pipeline Complete!")

Saving Results


Saving to ./llama_bengali_final...
Saved!

All results saved!

Files:
  - ./llama_bengali_final/ (model)
  - ./llama_bengali_checkpoints/ (checkpoints)
  - llama_experiments.csv
  - generated_responses.csv
  - training_history.json

🎉 Pipeline Complete!
